# Update the dataset boundary to include B2 Population centers
1. Download the metadata file from the latest version in TDEI system (GS_WSP_PG Project group and GS_WA_Proviso service)
2. Fetch the population centers included in that County
3. Make union of the population centers and the unincorporated boundary

In [50]:
import geopandas as gpd
import pandas as pd
pop_centers = gpd.read_file("../datasets/b2_coverage_pop_centers_reduced_precision.geojson")
counties = gpd.read_file("../datasets/WA_County_Boundaries.geojson") # May not be needed
pop_centers_with_county_df = pd.read_csv('pop_centers_with_county_names.csv')
pop_centers_with_county_df.head()

,County,PlaceName,OBJECTID,area_diff_sq_km,covers_osm_count
0,Grays Harbor,Aberdeen Gardens CDP,2,4.210018,1.0
1,Whatcom,Acme CDP,3,3.882192,1.0
2,Stevens,Addy CDP,4,0.625582,NaN
3,Stevens,Addy UGA,5,2.103468,NaN
4,Yakima,Ahtanum CDP,6,2.085586,NaN


In [51]:
# Get the county name from pop_centers_with_county_df based on OBJECTID
pop_centers.columns
county_map = pop_centers_with_county_df.set_index('OBJECTID')['County'].to_dict()
pop_centers['County'] = pop_centers['OBJECTID'].map(county_map)

In [78]:
pop_centers.head()

county_pop_centers = pop_centers[pop_centers['County'] == 'Grays Harbor']
print(len(county_pop_centers))

23


In [ ]:
# Get the county boundary from the geojson file
from shapely.ops import unary_union
county_name = 'graysharbor'
county_boundary_file = f'../ui-union/cleaned_boundaries/{county_name}_boundary.geojson'
county_gdf = gpd.read_file(county_boundary_file)
county_shape = county_gdf.iloc[0].geometry
county_pop_centers_union = county_pop_centers.union_all()
union_geometry = unary_union([county_shape,county_pop_centers_union])
# check if the union is valid
print(union_geometry.is_valid)
# save the union geometry to a geojson file
union_gdf = gpd.GeoDataFrame({'geometry': [union_geometry]}, crs=county_gdf.crs)
union_gdf.to_file(f'../ui-union/output_boundaries/{county_name}_ui_with_pop_centers.geojson', driver='GeoJSON')


True
